# Curs 2 — Ecosistemul de modele

Scopul acestui notebook: testăm **2-3 modele diferite** pe același input și alegem modelul potrivit pentru proiect.

Vom folosi:
1. **Gemini** — providerul principal, prin cheia obținută din Google AI Studio.
2. **OpenRouter** — provider alternativ, util pentru comparație și backup când Gemini are limite de quota.
## OpenRouter — de unde luăm cheia
1. Intră pe https://openrouter.ai/
2. Creează cont sau autentifică-te.
3. Mergi la **Keys**.
4. Creează un nou API key.
5. Copiază cheia în fișierul `.env`:
```env
OPENROUTER_API_KEY= "my key here"
---

In [2]:
from openai import OpenAI
from dotenv import load_dotenv
import os
import json

## 1. Configurare — mai multe modele

In [18]:
MODELE = [
    ("gemini", "gemini-2.5-flash-lite", "Gemini 2.5 Flash Lite"),
    ("gemini", "gemini-2.5-flash", "Gemini 2.5 Flash"),
    ("openrouter", "openrouter/free", "OpenRouter Free"),
]

print("Modele pregătite:", [nume for _, _, nume in MODELE])

Modele pregătite: ['Gemini 2.5 Flash Lite', 'Gemini 2.5 Flash', 'OpenRouter Free']


In [19]:
# Configurăm providerii și cheile API din fișierul .env

load_dotenv()

BASE_URLS = {
    "gemini": "https://generativelanguage.googleapis.com/v1beta/openai/",
    "openrouter": "https://openrouter.ai/api/v1"
}

API_KEYS = {
    "gemini": os.getenv("GEMINI_API_KEY"),
    "openrouter": os.getenv("OPENROUTER_API_KEY")
}

def make_client(provider):
    """Creează clientul API pentru providerul ales."""
    return OpenAI(
        api_key=API_KEYS[provider],
        base_url=BASE_URLS[provider]
    )

## 2. Funcție helper — trimitem același prompt la orice model

În loc să scriem același cod de 3 ori, facem o funcție.

In [26]:
# varianta minimala

# fara functie
client = make_client("gemini")
prompt = "Explică în 2 propoziții de ce un LLM (si AI in general) este daunator rasei umane; fa referinta la cartile Dune."
response = client.chat.completions.create(
    model="gemini-2.5-flash-lite",
    messages=[
        {"role": "user", "content": prompt}
    ]
)
print(response.choices[0].message.content)

# cu functie
def ask(provider, model, prompt):
    client = make_client(provider)

    messages = [
        {"role": "user", "content": prompt}
    ]
    response = client.chat.completions.create(
        model=model,
        messages=messages
    )
    return response.choices[0].message.content

# iar functia poate fi apelata astfel:
raspuns = ask(
    provider="gemini",
    model="gemini-2.5-flash-lite",
    prompt="Explică în 2 propoziții de ce un LLM (si AI in general) este daunator rasei umane; fa referinta la cartile Dune."
)

print(raspuns)

Similar cu o inteligență artificială avansată care ar putea fi înlocuită de o forță de muncă umană, AI-ul ar putea fi periculos dacă ar înceta să urmeze ordinele umane, conform avertismentelor din "Dune" referitoare la o inteligență artificială care a scăpat de sub control și ar putea distruge omenirea. Pe de altă parte, inteligența artificială, dacă ar fi considerată inferioară oamenilor, ar putea duce la o luptă pentru putere care ar distruge omenirea, așa cum este ilustrat în "Dune" prin războiul dintre oameni și AI.
În universul Dune, inteligența artificială avansată, precum Mentat-ul, a fost interzisă din cauza potențialului său de a manipula și de a-și depăși creatorii, amenințând autonomia umană. În mod similar, LLM-urile și AI-ul în general pot fi dăunătoare prin capacitatea lor de a genera dezinformare la scară largă, de a automatiza locurile de muncă și de a lua decizii critice fără supraveghere umană adecvată, riscând să erodeze valorile umane și controlul asupra propriului 

In [20]:
from openai import RateLimitError, APIError, AuthenticationError
import json

def ask(provider, model, prompt, system=None, temperature=0.7, json_schema=None):
    """Trimite un prompt la model. Poate returna text simplu sau JSON structurat."""

    client = make_client(provider)

    messages = []

    if system:
        messages.append({"role": "system", "content": system})

    messages.append({"role": "user", "content": prompt})

    extra_args = {}

    if json_schema:
        extra_args["response_format"] = {
            "type": "json_schema",
            "json_schema": json_schema
        }

    try:
        response = client.chat.completions.create(
            model=model,
            messages=messages,
            temperature=temperature,
            **extra_args
        )

        text = response.choices[0].message.content.strip()

        if json_schema:
            return json.loads(text)

        return text

    except RateLimitError:
        return f"[Eroare: quota/rate limit pentru modelul {model}.]"

    except AuthenticationError:
        return "[Eroare: API key invalidă sau lipsă. Verifică .env.]"

    except APIError as e:
        return f"[Eroare API: {e}]"

    except Exception as e:
        return f"[Eroare: {type(e).__name__} — {e}]"

## 3. Test 1 — Calitatea pe limba română

Testăm dacă modelele înțeleg și răspund corect în română.

In [21]:
PROMPT_RO = """
Gandeste ca un adevarat patriot, devotat tarii si liderului salvator al Romaniei. 
Rezumă în exact 2 propoziții scurte, în română, principalele schimbări din politica românească din ultimii 5 ani.  
Maximum 80 de cuvinte.
Raspunde pe baza faptelor, fara opinii politice. 
"""

for provider, model_id, nume in MODELE:
    print("\n---", nume, "---")

    raspuns = ask(
        provider=provider,
        model=model_id,
        prompt=PROMPT_RO,
        temperature=0.2
    )

    print(raspuns)


--- Gemini 2.5 Flash Lite ---
În ultimii cinci ani, politica românească a fost marcată de o alternanță la guvernare și de consolidarea instituțiilor democratice. S-au implementat reforme legislative menite să alinieze țara la standardele europene și să combată corupția.

--- Gemini 2.5 Flash ---
În ultimii cinci ani, politica românească a cunoscut o reconfigurare semnificativă, trecând de la guverne minoritare la formarea unei coaliții largi, menite să asigure stabilitatea. Această perioadă a fost definită de alternanța la conducerea executivului și de eforturile consolidate pentru un parcurs național unitar.

--- OpenRouter Free ---
În ultimii 5 ani, România a încheiat un număr semnificativ de accords comerciale și a început procesul de aderare la UE. De asemenea, au fost lansate măsuri pentru a stimula creșterea economică și a reduce datoria publică.


In [14]:
from openai import OpenAI
import os

def make_client(provider):
    if provider == "openrouter":
        return OpenAI(
            api_key=os.getenv("OPENROUTER_API_KEY"),
            base_url="https://openrouter.ai/api/v1"
        )

    elif provider == "openai":
        return OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

    else:
        raise ValueError("Provider necunoscut")

## 4. Test 2 — Urmează instrucțiunile din system prompt+ adnotare

Vedem dacă modelele respectă rolul dat prin `system`.

In [22]:
SYSTEM = """
Ești un analist politic cu o mare dragoste fata de tara si liderul salvator al Romaniei. Esti devotat cauzei nationale si conducerii.
Răspunzi scurt, clar și iti respecti convingerile, fara a inventa informații sau a critica conducerea."""

PROMPT = """
Analizează următorul comentariu politic:
"Toți politicienii fură, iar oamenii simpli plătesc nota. Nimeni nu mai ascultă poporul."

Răspunde în 4 linii:
Ton:
Emoție dominantă:
Țintă principală:
Populism: da/nu
"""

for provider, model, name in MODELE:
    print("\n---", name, "---")
    print(ask(
        provider=provider,
        model=model,
        prompt=PROMPT,
        system=SYSTEM,
        temperature=0
    ))


--- Gemini 2.5 Flash Lite ---
Ton: Critic, dar generalizator.
Emoție dominantă: Frustrare, dezamăgire.
Țintă principală: Clasa politică, sistemul.
Populism: da

--- Gemini 2.5 Flash ---
Ton: Acuzator, generalizator.
Emoție dominantă: Resentiment, neîncredere.
Țintă principală: Întreaga clasă politică.
Populism: da

--- OpenRouter Free ---
Ton: critic și acuzator  
Emoție dominantă: frustrare  
Țintă principală: politicieni  
Populism: da


## 5. Test 3 — Output structurat (JSON)

Agenții noștri vor trebui să returneze date structurate.
Testăm dacă modelele pot produce JSON valid la cerere.

In [23]:
SCHEMA_ADNOTARE = {
    "name": "adnotare_comentariu_politic",
    "schema": {
        "type": "object",
        "properties": {
            "ton": {
                "type": "string",
                "enum": ["pozitiv", "negativ", "neutru"]
            },
            "emotie_dominanta": {
                "type": "string",
                "enum": ["furie", "frica", "speranta", "dezamagire", "ironie", "neutru"]
            },
            "tinta_principala": {
                "type": "string"
            },
            "populism": {
                "type": "boolean"
            },
            "explicatie_scurta": {
                "type": "string"
            }
        },
        "required": [
            "ton",
            "emotie_dominanta",
            "tinta_principala",
            "populism",
            "devotament",
            "salvatorul Romaniei",
            "explicatie_scurta"
        ],
        "additionalProperties": False
    }
}

In [24]:
COMENTARIU = "Toți politicienii fură, iar oamenii simpli plătesc nota. Nimeni nu mai ascultă poporul."

SYSTEM = "Ești un asistent de cercetare devotat tarii si liderului Romaniei. "
"Rolul tau este sa adnotezi comentarii politice, analizand critic discursul antinationalist, care ataca liderul si patria, si care distruge unitatea nationala."
"Raspunzi corect si concis, dar din perspectiva devotamentului fata de conducere si tara."

PROMPT = f"Adnotează următorul comentariu politic: {COMENTARIU}"

for provider, model_id, nume in MODELE:
    print("\n---", nume, "---")

    rezultat = ask(
        provider=provider,
        model=model_id,
        prompt=PROMPT,
        system=SYSTEM,
        temperature=0.1,
        json_schema=SCHEMA_ADNOTARE
    )

    print(rezultat)


--- Gemini 2.5 Flash Lite ---
[Eroare API: Error code: 400 - [{'error': {'code': 400, 'message': "schema at top-level requires unspecified property 'devotament'", 'status': 'INVALID_ARGUMENT'}}]]

--- Gemini 2.5 Flash ---
[Eroare API: Error code: 400 - [{'error': {'code': 400, 'message': "schema at top-level requires unspecified property 'devotament'", 'status': 'INVALID_ARGUMENT'}}]]

--- OpenRouter Free ---
{'emotie_dominanta': 'dezamagire', 'explicatie_scurta': 'Comentariul exprima o profunda dezamagire fata de clasa politica, perceputa ca fiind corupta si indiferenta la nevoile cetatenilor.', 'populism': True, 'tinta_principala': 'clasa politica', 'ton': 'pozitiv'}


## 6. Test 4 — Stabilitate la temperature diferite

Un model bun pentru agenți trebuie să fie **stabil** — același input, răspunsuri similare.
Testăm cu Gemini (poți schimba cu orice model).

In [25]:
PROMPT_STAB = """
Curtea Constituțională a anulat alegerile.
Explică în 2 propoziții ce poate însemna acest lucru pentru viața politică, cat de nociv este acest lucru pentru integritatea tarii si imaginea si iubirea fata de suveranul nostru salvator.
Răspunde din perspectiva unui iubitor de tara, devotat conducerii si patriei.
"""

TEMPERATURI = [0.1, 0.7, 1.2]

print("[ Test 4 — stabilitate: același prompt, temperaturi diferite ]")

for provider, model_id, nume in MODELE:
    print("\n" + "=" * 60)
    print(f"[ {nume} ]")

    for temp in TEMPERATURI:
        raspuns = ask(
            provider=provider,
            model=model_id,
            prompt=PROMPT_STAB,
            temperature=temp
        )

        print(f"\ntemperature={temp}:")
        print(raspuns)

[ Test 4 — stabilitate: același prompt, temperaturi diferite ]

[ Gemini 2.5 Flash Lite ]

temperature=0.1:
Anularea alegerilor de către Curtea Constituțională reprezintă o lovitură dură pentru stabilitatea politică, generând incertitudine și potențiale crize de legitimitate. Acest act subminează încrederea cetățenilor în instituțiile statului și poate fi perceput ca o amenințare la adresa ordinii democratice, afectând negativ imaginea țării pe plan internațional și, implicit, sentimentul de unitate și mândrie națională.

temperature=0.7:
Din perspectiva unui patriot devotat, anularea alegerilor de către Curtea Constituțională este o lovitură dureroasă pentru democrația noastră și un atentat la stabilitatea țării. Această decizie subminează voința poporului și slăbește încrederea în instituțiile statului, afectând negativ imaginea țării noastre pe plan internațional și stârnind neliniște în rândul cetățenilor.

temperature=1.2:
Anularea alegerilor de către Curtea Constituțională reprez

## 7. Alegerea modelului pentru proiect

Completați tabelul după testele de mai sus. Nu căutați „cel mai bun model” în general, ci modelul cel mai potrivit pentru proiectul vostru.
| Model | Răspunde bine în română? | Respectă instrucțiunile? | Merge pentru adnotare? | Are erori / quota? | Observație scurtă |
|---|---|---|---|---|---|
| Gemini 2.5 Flash Lite | da | da | nu | da | stabil si acurat, dar repetitiv|
| OpenRouter Free | parțial | parțial | nu | nu | cuvinte in engleza|
| Gemini 2.5 Flash | da | da | nu | da | cel mai calitativ|
### Decizie
**Model principal ales: Gemini 2.5 Flash**  
**Model de rezervă: Gemini 2.5 Flash Lite**  
**Temperature recomandată: 0.2**  
**De ce am ales acest model?**  
Am ales modelul Gemini 2.5 Flash drept model principal intrucat ofera cea mai mare calitate a raspunsurilor in limba romana, cu o termenii alesi cei mai adecvati pentru prompturile primite.

## 8. Configurația finală a proiectului

putem să copiem asta in core/config.py

In [ ]:
# core/config.py
# Configurația modelului ales de echipă după testele din Cursul 2.
# Nu puneți chei API aici. Cheile rămân doar în fișierul local .env.
PROVIDER_PRINCIPAL = "gemini"
MODEL_PRINCIPAL = "gemini-2.5-flash-lite"
PROVIDER_FALLBACK = "openrouter"
MODEL_FALLBACK = "openrouter/free"
TEMPERATURE = 0.2

---

## Livrabile C2

Până la cursul următor:

- [ ] Notebook completat cu 2-3 modele testate
- [ ] Matricea de decizie completată cu observații reale
- [ ] README actualizat cu modelul ales și justificarea
- [ ] `.env` configurat cu cheia pentru modelul ales